# RAND HRS Silver Table Generation Specification

---

# 1. Document Information

| Property | Value |
|----------|-------|
| Document Name | |
| Document Version | 1.0 |
| Author | Pete Perez |
| Last Updated | |
| Databricks Runtime | 15.x |
| Catalog | |
| Schema | slv_cdm_hrs |
| Layer | Silver |

---

# 2. Table Overview

## Target Table

| Property | Value |
|----------|-------|
| Table Name | |
| Business Subject | |
| Description | |

---

## Business Purpose

Describe the business purpose of the table.

Example:

> The Health table stores respondent health-related observations collected during each HRS survey wave.

---

# 3. Source Information

## Source Dataset

| Property | Value |
|----------|-------|
| Source System | RAND HRS |
| Source Catalog | dev_catalog |
| Source Schema | brz_raw_hrs |
| Source Table | randhrs1992_2022v1 |
| Source Grain | One row per respondent (HHIDPN) |

---

## RAND Codebook Reference

| Property | Value |
|----------|-------|
| Section | Section B: Health |
| Subsection | Self-reportofhealth |
| Page Numbers | 246-250|
| Notes | |

---

# 4. Target Data Model

## Table Grain

**One row represents one respondent for one HRS survey wave.**

---

## Parent Tables

### hrs_respondent

| Property | Value |
|----------|-------|
| Primary Key | respondent_id |

Relationship

```
hrs_respondent (1)
        │
        └──────────────< Target Table (Many)
```

---

### hrs_wave

| Property | Value |
|----------|-------|
| Primary Key | wave_id |

Relationship

```
hrs_wave (1)
      │
      └──────────────< Target Table (Many)
```

---

## Target Table

| Property | Value |
|----------|-------|
| Primary Key | <table_name>_id |
| Foreign Key | respondent_id |
| Foreign Key | wave_id |

---

## Business Key

Although the table contains a surrogate primary key, the logical business key is:

```
respondent_id
+
wave_id
```

A respondent should have only one record in this subject area for each survey wave.

---

# 5. Source Variable Mapping

Complete this table directly from the RAND Codebook.

| Wave | Source Variable | Variable Label | RAND Type | Target Column | Databricks Type | Transformation |
|------|-----------------|----------------|-----------|---------------|-----------------|----------------|
| | | | | | | |

Example

| Wave | Source Variable | Variable Label | Target Column |
|------|-----------------|----------------|---------------|
| R1 | R1SHLT | Self Rated Health | self_rated_health |
| R2 | R2SHLT | Self Rated Health | self_rated_health |
| R3 | R3SHLT | Self Rated Health | self_rated_health |

---

# 6. Target Columns

| Column Name | Data Type | Nullable | Description |
|-------------|-----------|----------|-------------|
| | | | |

---

# 7. Data Type Mapping

| RAND Type | Databricks Type |
|-----------|-----------------|
| Numeric | INT |
| Decimal | DECIMAL |
| Character | STRING |
| Date | DATE |

---

# 8. Missing Value Handling

Convert RAND missing value codes to SQL NULL.

| RAND Value | Meaning |
|------------|---------|
| -1 | Don't Know |
| -7 | Refused |
| -8 | Not Applicable |
| -9 | Missing |

Transformation Rule

```sql
CASE
    WHEN value IN (-1,-7,-8,-9)
        THEN NULL
    ELSE value
END
```

---

# 9. Business Rules

Document all transformation rules.

## Rule 1

Description

Transformation SQL

---

## Rule 2

Description

Transformation SQL

---

# 10. Derived Columns

| Column | Description | Logic |
|---------|-------------|-------|
| | | |

---

# 11. DDL Requirements

Generate a Databricks SQL DDL script that includes:

- Delta Table
- Unity Catalog
- Surrogate Primary Key
- Foreign Key to hrs_respondent
- Foreign Key to hrs_wave
- Audit Columns
- Column Comments

Standard Columns

| Column |
|---------|
| created_timestamp |
| updated_timestamp |
| created_by |
| updated_by |

---

# 12. DML Requirements

Generate a Databricks SQL load script.

Load Pattern

```
RAND HRS Source
        │
        ▼
Lookup respondent_id
        │
        ▼
Lookup wave_id
        │
        ▼
Insert into Target Table
```

The load should:

- Lookup respondent_id using HHIDPN
- Lookup wave_id using the survey wave
- Convert missing values to NULL
- Populate audit columns
- Load one record per respondent per survey wave

---

# 13. Validation Requirements

Generate validation SQL for:

## Row Count

```sql
SELECT COUNT(*)
FROM <table>;
```

---

## Duplicate Business Key

```sql
SELECT
    respondent_id,
    wave_id,
    COUNT(*)
FROM <table>
GROUP BY
    respondent_id,
    wave_id
HAVING COUNT(*) > 1;
```

---

## Null Foreign Keys

```sql
SELECT *
FROM <table>
WHERE respondent_id IS NULL
   OR wave_id IS NULL;
```

---

## Referential Integrity

Validate all foreign keys.

---

# 14. Deliverables

Generate the following artifacts.

```
sql/

    ddl/

        <table_name>.sql

    dml/

        load_<table_name>.sql

validation/

    validate_<table_name>.sql

docs/

    <table_name>_data_dictionary.md
```

---

# 15. Coding Standards

## Naming

Primary Key

```
<table_name>_id
```

Foreign Keys

```
respondent_id

wave_id
```

Audit Columns

```
created_timestamp

updated_timestamp

created_by

updated_by
```

---

## Parent Tables

```
hrs_respondent
    respondent_id (PK)

hrs_wave
    wave_id (PK)
```

---

## Child Table Pattern

Every subject-area table follows this model.

```
                 hrs_respondent
                respondent_id (PK)
                        │
                        │
                        │
                        ▼
                 Subject Area Table
               ----------------------
               <table_name>_id (PK)

               respondent_id (FK)

               wave_id (FK)

               Business Attributes...
                        ▲
                        │
                        │
                   hrs_wave
                   wave_id (PK)
```

Examples

- hrs_health
- hrs_income
- hrs_employment
- hrs_insurance
- hrs_pensions
- hrs_family
- hrs_functional_limitations

---

# 16. Prompt to ChatGPT

After completing Sections 1–10, use the following prompt:

> Using this specification, generate:
>
> 1. A production-ready Databricks SQL DDL script.
> 2. A production-ready Databricks SQL DML load script.
> 3. Validation SQL.
> 4. A data dictionary.
> 5. Any assumptions or recommendations.
>
> Target Runtime: Databricks Runtime 15.x
> Catalog: Unity Catalog
> Layer: Silver

# RAND HRS Silver Layer Table Generation Specification

## Instructions

Complete this specification for each Silver layer table.

This document will be used to generate:

1. Databricks SQL DDL script
2. Databricks SQL DML load script
3. Data validation SQL
4. Data dictionary documentation
5. ETL assumptions and recommendations

Target Platform:

- Databricks Runtime: 15.x
- Catalog: Unity Catalog
- Data Layer: Silver
- Database Type: Delta Lake

---

# Generation Prompt

When this specification is complete, generate the following:

## Required Deliverables

### 1. DDL Script

Create:

```
sql/ddl/hrs_health.sql
```

Requirements:

- Databricks SQL syntax
- Delta table
- Unity Catalog compatible
- Surrogate primary key
- Foreign key to hrs_respondent
- Foreign key to hrs_wave
- Column comments
- Audit columns
- Proper data types
- Naming standards

---

### 2. DML Load Script

Create:

```
sql/dml/load_hrs_health.sql
```

Requirements:

- Load from Bronze RAND HRS dataset
- Resolve respondent_id from hrs_respondent
- Resolve wave_id from hrs_wave
- Convert RAND missing values to NULL
- Handle all available waves
- Produce one row per respondent per wave
- Include audit columns

---

### 3. Validation SQL

Create:

```
sql/validation/validate_hrs_health.sql
```

Include:

- Row count validation
- Duplicate business key checks
- NULL foreign key checks
- Referential integrity checks
- Source-to-target reconciliation

---

### 4. Documentation

Create:

```
docs/hrs_health_data_dictionary.md
```

Include:

- Table purpose
- Grain
- Column descriptions
- Source mappings
- Transformation rules

---

# 1. Document Information

| Attribute | Value |
|---|---|
| Document Name | |
| Version | 1.0 |
| Author | |
| Date Created | |
| Last Updated | |
| Target Table | |
| Target Catalog | |
| Target Schema | slv_cdm_hrs |
| Data Layer | Silver |

---

# 2. Target Table Definition

## Table Name

Example:

```
hrs_health
```

---

## Business Purpose

Describe why this table exists.

Example:

```
Stores respondent health characteristics collected during each HRS survey wave.
```

---

## Table Grain

IMPORTANT:

The Silver model uses:

```
ONE ROW PER RESPONDENT PER SURVEY WAVE
```

Describe the grain:

Example:

```
One row represents one respondent's health information for one HRS survey wave.
```

---

# 3. Parent Table Relationships

## Parent Table 1

### Respondent Dimension

Table:

```
hrs_respondent
```

Primary Key:

```
respondent_id
```

Natural Key:

```
HHIDPN
```

Relationship:

```
hrs_respondent (1)
        |
        |
        +-------------< target_table (many)
```

---

## Parent Table 2

### Wave Dimension

Table:

```
hrs_wave
```

Primary Key:

```
wave_id
```

Relationship:

```
hrs_wave (1)
        |
        |
        +-------------< target_table (many)
```

---

# 4. Target Table Keys

## Surrogate Primary Key

Pattern:

```
health_id
```

Example:

```
health_id
```

---

## Foreign Keys

| Column | Parent Table | Parent Key |
|---|---|---|
| respondent_id | hrs_respondent | respondent_id |
| wave_id | hrs_wave | wave_id |

---

## Business Key

The logical unique identifier:

```
respondent_id + wave_id
```

Business Rule:

```
A respondent may have only one record per subject area per survey wave.
```

---

# 5. RAND HRS Source Information

## Source Dataset

| Attribute | Value |
|---|---|
| Source System | RAND HRS |
| Source Catalog | dev_catalog |
| Source Schema | brz_raw_hrs |
| Source Table | randhrs1992_2022v1 |
| Source Grain | One row per respondent |

Source Path
```
dev_catalog.brz_raw_hrs.randhrs1992_2022v1
```

---

# 6. RAND HRS Codebook Reference

## Section

```
Section B: Health
```

---

## Subsection

```
Self-reportofhealth
```

---

## Codebook Pages

```
Pages: 246 - 250
```

---

## Codebook Notes

```

```

---

# 7. Source Variable Mapping

Copy the RAND codebook variable matrix here.

| Wave | Variable | Label | Type | Target Column | Databricks Type | Transformation |
|---|---|---|---|---|---|---|
| | | | | | | |
1 R1SHLT R1SHLT:W1Self-reportofhealth Categ
2 R2SHLT R2SHLT:W2Self-reportofhealth Categ
3 R3SHLT R3SHLT:W3Self-reportofhealth Categ
4 R4SHLT R4SHLT:W4Self-reportofhealth Categ
5 R5SHLT R5SHLT:W5Self-reportofhealth Categ
6 R6SHLT R6SHLT:W6Self-reportofhealth Categ
7 R7SHLT R7SHLT:W7Self-reportofhealth Categ
8 R8SHLT R8SHLT:W8Self-reportofhealth Categ
9 R9SHLT R9SHLT:W9Self-reportofhealth Categ
10 R10SHLT R10SHLT:W10Self-reportofhealth Categ
11 R11SHLT R11SHLT:W11Self-reportofhealth Categ
12 R12SHLT R12SHLT:W12Self-reportofhealth Categ
13 R13SHLT R13SHLT:W13Self-reportofhealth Categ
14 R14SHLT R14SHLT:W14Self-reportofhealth Categ
15 R15SHLT R15SHLT:W15Self-reportofhealth Categ
16 R16SHLT R16SHLT:W16Self-reportofhealth Categ
1 S1SHLT S1SHLT:W1Self-reportofhealth Categ
2 S2SHLT S2SHLT:W2Self-reportofhealth Categ
3 S3SHLT S3SHLT:W3Self-reportofhealth Categ
4 S4SHLT S4SHLT:W4Self-reportofhealth Categ
5 S5SHLT S5SHLT:W5Self-reportofhealth Categ
6 S6SHLT S6SHLT:W6Self-reportofhealth Categ
7 S7SHLT S7SHLT:W7Self-reportofhealth Categ
8 S8SHLT S8SHLT:W8Self-reportofhealth Categ
9 S9SHLT S9SHLT:W9Self-reportofhealth Categ
10 S10SHLT S10SHLT:W10Self-reportofhealth Categ
11 S11SHLT S11SHLT:W11Self-reportofhealth Categ
12 S12SHLT S12SHLT:W12Self-reportofhealth Categ
13 S13SHLT S13SHLT:W13Self-reportofhealth Categ
14 S14SHLT S14SHLT:W14Self-reportofhealth Categ
15 S15SHLT S15SHLT:W15Self-reportofhealth Categ
16 S16SHLT S16SHLT:W16Self-reportofhealth Categ

Example:

| Wave | Source Variable | Variable Label | Target Column |
|-|-|-|-|
| R1 | R1SHLT | Self Rated Health | self_rated_health |
| R2 | R2SHLT | Self Rated Health | self_rated_health |
| R3 | R3SHLT | Self Rated Health | self_rated_health |


---

# 8. Target Column Definitions

These columns will be created in the Silver table.

| Column Name | Data Type | Nullable | Description |
|---|---|---|---|
| | | | |

---

# 9. Data Type Mapping

Document conversions.

| RAND Type | Databricks Type |
|---|---|
| Numeric | INT |
| Decimal | DECIMAL |
| Character | STRING |
| Date | DATE |

---

# 10. RAND Missing Value Handling

Default RAND HRS missing codes:

| Value | Description |
|---|---|
| -1 | Don't Know |
| -7 | Refused |
| -8 | Not Applicable |
| -9 | Missing |

Transformation:

```sql
CASE
    WHEN value IN (-1,-7,-8,-9)
        THEN NULL
    ELSE value
END
```

Additional rules:

```

```

---

# 11. Business Transformation Rules

Document all transformations.

## Rule 1

Description:

```

```

SQL Logic:

```sql

```

---

## Rule 2

Description:

```

```

SQL Logic:

```sql

```

---

# 12. Wave Processing Rules

The RAND dataset contains wave-specific columns:

Example:

```
R1VARIABLE
R2VARIABLE
R3VARIABLE
...
R16VARIABLE
```

The load process should:

```
UNPIVOT wave-specific variables

into:

respondent_id
wave_id
subject attributes
```

Additional wave rules:

```

```

---

# 13. Load Strategy

Select:

```
[ ] Full Refresh

[ ] Incremental

[ ] Merge / Upsert

[ ] Append Only
```

---

# 14. Audit Requirements

Include:

| Column | Purpose |
|---|---|
| created_timestamp | ETL audit |
| updated_timestamp | ETL audit |
| created_by | Process tracking |
| updated_by | Process tracking |

---

# 15. Data Quality Requirements

## Duplicate Check

Business key:

```
respondent_id + wave_id
```

Expected Result:

```
0 duplicates
```

---

## Referential Integrity

Validate:

```
respondent_id exists in hrs_respondent

wave_id exists in hrs_wave
```

---

## Null Requirements

Required columns:

```

```

---

# 16. Analyst Notes

Document assumptions:

```

```

---

# END SPECIFICATION

After completing this document, generate:

1. DDL SQL
2. DML SQL
3. Validation SQL
4. Data Dictionary
5. ETL Recommendations

Follow the established RAND HRS Silver Layer modeling pattern:

```
                    hrs_respondent
                   respondent_id PK
                           |
                           |
                           v

                    Subject Table
              --------------------------
              <subject>_id PK

              respondent_id FK

              wave_id FK

              Subject Attributes


                           ^
                           |

                       hrs_wave
                         wave_id PK
```